In [1]:
from qiskit import ClassicalRegister, QuantumRegister, QuantumCircuit, transpile
from qiskit_aer import AerSimulator

from LogicalQ.Logical import LogicalCircuit
from LogicalQ.Library.QECCs import steane_code

from LogicalQ.NoiseModel import (
    construct_noise_model,
    construct_noise_model_QuantinuumH1_1,
    construct_noise_model_QuantinuumH2_1,
    construct_noise_model_QuantinuumH2_2,
)

from LogicalQ.Library.HardwareModels import (
    hardware_model_depolarizing,
    hardware_model_Quantinuum_H1_1,
    hardware_model_Quantinuum_H2_1,
    hardware_model_Quantinuum_H2_2,
)

from LogicalQ.Benchmarks import (
    n_qubit_ghz_generation,
    quantum_teleportation,
    mirror_benchmarking,
    randomized_benchmarking,
    quantum_volume,
)

from LogicalQ.Experiments import (
    execute_circuits,
    circuit_scaling_experiment,
    noise_scaling_experiment,
    qec_cycle_efficiency_experiment,
    qec_cycle_circuit_scaling_experiment,
    qec_cycle_noise_scaling_experiment,
)

from LogicalQ.Analysis import (
    circuit_scaling_bar3d,
    noise_scaling_scatter,
    qec_cycle_efficiency_scatter,
)

# from KS import KogutSusskindHamiltonian
from TPSU2 import TwoPlaquetteSU2

# Circuit Scaling

In [2]:
# Construct circuit factories as circuit inputs
physical_circuit_factory = lambda n_qubits, circuit_length : mirror_benchmarking(n_qubits=n_qubits, circuit_length=circuit_length, measure=True)

def log_mirror_benchmarking(n_qubits, circuit_length):
    pqc = physical_circuit_factory(n_qubits, circuit_length)
    lqc = LogicalCircuit.from_physical_circuit(pqc, **steane_code)
    for _ in range(1): lqc.append_qec_cycle()
    return lqc

logical_circuit_factory = log_mirror_benchmarking

# Construct noise model factories as noise model inputs
noise_model_factory = construct_noise_model_QuantinuumH2_1()

## Serial

In [3]:
phys_data_serial = circuit_scaling_experiment(
    physical_circuit_factory, noise_model_factory,
    min_n_qubits=1, max_n_qubits=1,
    min_circuit_length=1, max_circuit_length=16,
    backend="aer_simulator", hardware_model=hardware_model_Quantinuum_H2_1, shots=1E5,
    with_mp=False
)

NameError: name 'NoiseModel' is not defined

In [ ]:
log_data_serial = circuit_scaling_experiment(
    logical_circuit_factory, noise_model_factory,
    min_n_qubits=1, max_n_qubits=1,
    min_circuit_length=1, max_circuit_length=15,
    backend="aer_simulator", hardware_model=hardware_model_Quantinuum_H2_1, coupling_map=None, shots=1E5,
    with_mp=False
)

In [ ]:
circuit_scaling_bar3d(phys_data_serial, title="Physical circuit scaling (serial)", show=True)

In [ ]:
circuit_scaling_bar3d(log_data_serial, title="Logical circuit scaling (serial)", show=True)

## With MP

In [ ]:
phys_data_mp = circuit_scaling_experiment(
    physical_circuit_factory,
    noise_model_factory,
    min_n_qubits=1,
    max_n_qubits=8,
    min_circuit_length=1,
    max_circuit_length=32,
    shots=1E5,
    with_mp=True
)

In [ ]:
circuit_scaling_bar3d(phys_data_mp, title="Physical circuit scaling (mp)", show=True)

In [ ]:
log_data_mp = circuit_scaling_experiment(
    logical_circuit_factory,
    noise_model_factory,
    min_n_qubits=1, max_n_qubits=1,
    min_circuit_length=1, max_circuit_length=32,
    shots=1E2,
    with_mp=True
)

In [ ]:
circuit_scaling_bar3d(log_data_mp, circuit_type="logical", qecc=steane_code, title="Logical circuit scaling (mp)", show=True)

# Noise Model Scaling

In [ ]:
# Sweep 3x3x3 = 27 combinations of noise values.
error_scan_keys = ["amplitude_damping_error_1q", "depolarizing_error_2q", "readout_error_0|1"]
error_scan_val_lists = [
    [0.0, 2e-4, 1e-3],   # 1q amplitude damping error
    [0.0, 5e-3, 2e-2],   # 2q depolarizing error
    [0.0, 5e-3, 5e-2],   # readout 0->1
]

qc = mirror_benchmarking(n_qubits=1, circuit_length=4)
lqc = LogicalCircuit.from_physical_circuit(qc, **steane_code)

qc.measure_all()
lqc.measure_all()

base_noise_model = construct_noise_model_QuantinuumH1_1()

## Logical circuit

In [ ]:
noise_data_logical_all_true = noise_scaling_experiment(
    circuit_input=lqc,
    noise_model_input=base_noise_model,
    error_scan_keys=error_scan_keys,
    error_scan_val_lists=error_scan_val_lists,
    compute_exact=True,
    shots=1024,
    with_mp=True
)

In [ ]:
noise_data_logical_all_false = noise_scaling_experiment(
    circuit_input=lqc,
    noise_model_input=base_noise_model,
    error_scan_keys=error_scan_keys,
    error_scan_val_lists=error_scan_val_lists,
    compute_exact=False,
    shots=1,
    with_mp=False
)

In [ ]:
noise_model_scaling_bar(noise_data_logical_all_true, scan_keys=error_scan_keys, separate_plots=True)

In [ ]:
noise_model_scaling_bar(noise_data_logical_all_false, scan_keys=error_scan_keys, separate_plots=False)

## Physical circuit

In [ ]:
noise_data_phys_all_true = noise_scaling_experiment(
    circuit_input=qc,
    noise_model_input=base_noise_model,
    error_scan_keys=error_scan_keys,
    error_scan_val_lists=error_scan_val_lists,
    compute_exact=True,
    shots=1024,
    with_mp=True
)

In [ ]:
noise_data_phys_all_false = noise_scaling_experiment(
    circuit_input=qc,
    noise_model_input=base_noise_model,
    error_scan_keys=error_scan_keys,
    error_scan_val_lists=error_scan_val_lists,
    compute_exact=True,
    shots=256,
    with_mp=False
)

In [ ]:
noise_model_scaling_bar(noise_data_phys_all_true, scan_keys=error_scan_keys, separate_plots=True)

In [ ]:
noise_model_scaling_bar(noise_data_phys_all_false, scan_keys=error_scan_keys, separate_plots=False)

# QEC Cycle Efficiency

In [ ]:
# Insert varying number of QEC cycles and see how fidelity behaves
constraint_scan_keys = ["effective_threshold", "circuit_depth_logical_qubit"]
constraint_scan_val_lists = [[1e-3], list(range(1,4+1))]

qc = mirror_benchmarking(n_qubits=1, circuit_length=4)

# Concrete noise model (base Quantinuum H1-1)
base_noise_model = construct_noise_model_QuantinuumH1_1()

## Serial

In [ ]:
qec_eff_data_serial = qec_cycle_efficiency_experiment(
    circuit_input=qc,
    noise_model_input=base_noise_model,
    qecc=steane_code,
    constraint_scan_keys=constraint_scan_keys,
    constraint_scan_val_lists=constraint_scan_val_lists,
    method="density_matrix",
    shots=256,
    with_mp=False
)

In [ ]:
qec_cycle_efficiency_bar(qec_eff_data_serial)

## With MP

In [ ]:
qec_eff_data_mp = qec_cycle_efficiency_experiment(
    circuit_input=qc,
    noise_model_input=base_noise_model,
    qecc=steane_code,
    constraint_scan_keys=constraint_scan_keys,
    constraint_scan_val_lists=constraint_scan_val_lists,
    method="density_matrix",
    shots=256,
    with_mp=True
)

In [ ]:
qec_cycle_efficiency_bar(qec_eff_data_mp)

# QEC Cycle Circuit Scaling

In [2]:
x = 1
dt = 0.1

def physical_circuit_factory(n_qubits, circuit_length):
    qc = TwoPlaquetteSU2(x, dt, circuit_length, measure=True, mitigation=False)
    return qc

In [3]:
physical_circuit_factory(5, 1).draw()

┌───┐┌───────────┐┌─────────────┐┌───┐ ┌───────────┐ ┌────────────┐»
   q_0: ┤ X ├┤ Ry(-0.05) ├┤ Rz(-0.0375) ├┤ X ├─┤ Ry(-0.15) ├─┤ Rz(-0.225) ├»
        └─┬─┘└───────────┘└─────────────┘└─┬─┘┌┴───────────┴┐└┬──────────┬┘»
   q_1: ──■────────────────────────────────■──┤ Rz(-0.1125) ├─┤ Ry(-0.3) ├─»
                                              └─────────────┘ └──────────┘ »
meas: 2/═══════════════════════════════════════════════════════════════════»
                                                                           »
«                               ┌───────────┐ ┌───┐┌─────────────┐┌───────────┐»
«   q_0: ──■────────────────■───┤ Ry(-0.15) ├─┤ X ├┤ Rz(-0.0375) ├┤ Ry(-0.05) ├»
«        ┌─┴─┐┌──────────┐┌─┴─┐┌┴───────────┴┐└─┬─┘└─────────────┘└───────────┘»
«   q_1: ┤ X ├┤ Ry(-0.1) ├┤ X ├┤ Rz(-0.1125) ├──■──────────────────────────────»
«        └───┘└──────────┘└───┘└─────────────┘                                 »
«meas: 2/══════════════════════════════════════════════════════════════════════»
«                                                                              »
«        ┌───┐ ░ ┌─┐   
«   q_0: ┤ X ├─░─┤M├───
«        └─┬─┘ ░ └╥┘┌─┐
«   q_1: ──■───░──╫─┤M├
«              ░  ║ └╥┘
«meas: 2/═════════╩══╩═
«                 0  1

In [4]:
effective_threshold = 1E-2
constraint_model = {
    "effective_threshold": effective_threshold,
    "cost_ops_1q": effective_threshold/2,
    # "cost_ops_1q": effective_threshold/4,
    # "cost_ops_pauli_1q": effective_threshold/4,
    # "cost_ops_clifford_1q": effective_threshold/4,
    # "cost_ops_nonclifford_1q": effective_threshold/4
}

In [5]:
data = qec_cycle_circuit_scaling_experiment(
    physical_circuit_factory,
    qecc=steane_code,
    constraint_model=constraint_model,
    min_n_qubits=1, max_n_qubits=1,
    min_circuit_length=1, max_circuit_length=2,
    backend="aer_simulator",
    hardware_model=hardware_model_depolarizing,
    # coupling_map="fully_coupled",
    method="matrix_product_state",
    shots=1E2,
    with_mp=False
)

TranspilerError: 'The circuit requires a connection between physical qubits [Qubit(3), Qubit(7)] for cx'

In [ ]:
qec_cycle_efficiency_scatter(data)

# QEC Cycle Noise Scaling

In [ ]:
error_scan_keys = ["cost_ops"]
error_scan_val_lists = [[0.0, effective_threshold/2]]

In [ ]:
qec_noise_scaling_serial = qec_cycle_noise_scaling_experiment(
    circuit_input=qc,
    noise_model_input=base_noise_model,
    qecc=steane_code,
    constraint_scan_keys=constraint_scan_keys,
    constraint_scan_val_lists=constraint_scan_val_lists,
    error_scan_keys=qec_noise_keys,
    error_scan_val_lists=qec_noise_vals,
    save_filename=None,
    with_mp=False
)

In [ ]:
qec_noise_scaling_mp = qec_cycle_noise_scaling_experiment(
    circuit_input=qc,
    noise_model_input=base_noise_model,
    qecc=steane_code,
    constraint_scan_keys=constraint_scan_keys,
    constraint_scan_val_lists=constraint_scan_val_lists,
    error_scan_keys=qec_noise_keys,
    error_scan_val_lists=qec_noise_vals,
    save_filename=None,
    with_mp=True
)